# Causal Self-Attention

Implements and tests causal self-attention with lower-triangular masking and dropout.


Imports PyTorch modules for neural network layers and functional operations.


In [1]:
import torch 
import math 
import torch.nn as nn 
import torch.nn.functional as F

Implements causal self-attention with triangular masking to prevent tokens from attending to future positions.


In [9]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        query = self.W_query(x)
        key = self.W_key(x)
        value = self.W_value(x)

        d_k = key.size(-1)
        atten_score = query @ key.transpose(-2, -1)

        seq_len = atten_score.size(-1)

        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).bool()

        masked_atten_score = atten_score.masked_fill(~mask, float('-inf'))

        atten_weights = torch.softmax(masked_atten_score / math.sqrt(d_k), dim=-1)

        torch.manual_seed(789)
        drop_atten_weights = F.dropout(atten_weights, p=0.1)

        context = drop_atten_weights @ value
        return context

Tests causal attention on a sample tensor and confirms output dimensions.


In [10]:
d_in = 4
d_out = 2
torch.manual_seed(789)

inputs = torch.randn(1, 3, d_in)

sa = CausalAttention(d_in, d_out)
print(sa(inputs))

tensor([[[-0.4684,  0.0188],
         [-0.3012, -0.2176],
         [-0.1737, -0.1098]]], grad_fn=<UnsafeViewBackward0>)
